# 01_b_rework_preprocess: Marketing-Driven Cleaning and Preprocessing

## Why this notebook exists

This notebook rebuilds preprocessing with a **marketing strategy perspective** instead of only technical merge-and-fillna steps.

Core principles:
- Treat each dataset (`portfolio`, `profile`, `transcript`) separately first because each captures different business meaning.
- Preserve signal-rich unknowns (for example unknown gender/income behavior can be a real customer segment).
- Build the target using event sequence logic (received -> viewed -> completed) and offer timing constraints.
- Use imputation with segment logic and explicit indicator flags, not blind global fills.
- Engineer features that can support campaign strategy and customer segmentation.

## Workflow Overview

1. Load raw datasets with robust project-relative paths.
2. Clean `portfolio` (offer design features + channel structure).
3. Clean `profile` (customer demographic quality + tenure).
4. Clean `transcript` (event normalization + transaction extraction).
5. Build person-offer journey table with timing logic for response success.
6. Merge cleaned datasets.
7. Apply marketing-aware missing-value strategy with flags.
8. Add campaign and customer features.
9. Export model-ready dataset.

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

portfolio_path = DATA_DIR / 'portfolio.json'
profile_path = DATA_DIR / 'profile.json'
transcript_path = DATA_DIR / 'transcript.json'

for p in [portfolio_path, profile_path, transcript_path]:
    if not p.exists():
        raise FileNotFoundError(f'Expected input file not found: {p}')

portfolio_raw = pd.read_json(portfolio_path, orient='records', lines=True)
profile_raw = pd.read_json(profile_path, orient='records', lines=True)
transcript_raw = pd.read_json(transcript_path, orient='records', lines=True)

print('Loaded raw data:')
print('portfolio:', portfolio_raw.shape)
print('profile:', profile_raw.shape)
print('transcript:', transcript_raw.shape)

Loaded raw data:
portfolio: (10, 6)
profile: (17000, 5)
transcript: (306534, 4)


## Step 1) Clean `portfolio` (Offer Design Table)

### Marketing rationale

`portfolio` defines campaign design: reward, difficulty, duration, type, and channels. These are direct levers of campaign performance.

Cleaning strategy:
- Normalize ID naming (`id` -> `offer_id`) for clean joins.
- Keep structured offer attributes and add channel indicators.
- Keep `email` channel even if low variance for auditability, then drop later only if truly constant in modeling.
- Derive offer economics features such as reward-to-difficulty ratio and reward-per-day intensity.

In [25]:
def clean_portfolio(portfolio_df: pd.DataFrame) -> pd.DataFrame:
    p = portfolio_df.copy()

    required_cols = ['id', 'offer_type', 'difficulty', 'reward', 'duration', 'channels']
    missing = [c for c in required_cols if c not in p.columns]
    if missing:
        raise KeyError(f'portfolio is missing required columns: {missing}')

    p = p.rename(columns={'id': 'offer_id'})
    p['offer_id'] = p['offer_id'].astype(str)
    p['offer_type'] = p['offer_type'].astype(str).str.lower().str.strip()

    # Safe channels handling for robust one-hot encoding
    p['channels'] = p['channels'].apply(lambda x: x if isinstance(x, list) else [])
    unique_channels = sorted(pd.Series(p['channels'].explode().dropna().unique()).astype(str).tolist())
    for ch in unique_channels:
        p[f'channel_{ch}'] = p['channels'].apply(lambda lst: int(ch in lst))

    p['channel_count'] = p[[f'channel_{ch}' for ch in unique_channels]].sum(axis=1) if unique_channels else 0

    # Offer economics features
    p['reward_to_difficulty'] = np.where(p['difficulty'] > 0, p['reward'] / p['difficulty'], 0.0)
    p['reward_per_day'] = np.where(p['duration'] > 0, p['reward'] / p['duration'], 0.0)

    # One-hot offer type for model-ready table
    p = pd.get_dummies(p, columns=['offer_type'], prefix='offer_type', dtype=int)

    return p

portfolio_clean = clean_portfolio(portfolio_raw)
print('portfolio_clean:', portfolio_clean.shape)
portfolio_clean.head()

portfolio_clean: (10, 15)


,reward,channels,difficulty,duration,offer_id,channel_email,channel_mobile,channel_social,channel_web,channel_count,reward_to_difficulty,reward_per_day,offer_type_bogo,offer_type_discount,offer_type_informational
0,10,"[email, mobile, social]",10,7,ae264e3637204a6fb9bb56bc8210ddfd,1,1,1,0,3,1.00,1.428571,1,0,0
1,10,"[web, email, mobile, social]",10,5,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1,4,1.00,2.000000,1,0,0
2,0,"[web, email, mobile]",0,4,3f207df678b143eea3cee63160fa8bed,1,1,0,1,3,0.00,0.000000,0,0,1
3,5,"[web, email, mobile]",5,7,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,0,1,3,1.00,0.714286,1,0,0
4,5,"[web, email]",20,10,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,0,0,1,2,0.25,0.500000,0,1,0


## Step 2) Clean `profile` (Customer Table)

### Marketing rationale

`profile` is customer context. We should avoid over-smoothing this table because unknown values can represent meaningful segments (privacy-sensitive or low-engagement users).

Cleaning strategy:
- Rename `id` to `person` for alignment with transcript.
- Convert member start date and derive `days_since_registration` (tenure proxy).
- Treat age `118` as missing.
- Impute income using hierarchical segment medians (gender x age_band), then fallback to global median.
- Add explicit missing/imputation flags to preserve behavioral signal.

In [26]:
def clean_profile(profile_df: pd.DataFrame) -> pd.DataFrame:
    pr = profile_df.copy()

    required_cols = ['id', 'age', 'gender', 'income', 'became_member_on']
    missing = [c for c in required_cols if c not in pr.columns]
    if missing:
        raise KeyError(f'profile is missing required columns: {missing}')

    pr = pr.rename(columns={'id': 'person'})
    pr['person'] = pr['person'].astype(str)

    # Parse registration date and create tenure feature
    member_date = pd.to_datetime(pr['became_member_on'].astype(str), format='%Y%m%d', errors='coerce')
    reference_date = member_date.max()
    pr['days_since_registration'] = (reference_date - member_date).dt.days

    # Quality flags
    pr['age_was_118'] = (pr['age'] == 118).astype(int)
    pr['income_missing_before_impute'] = pr['income'].isna().astype(int)
    pr['gender_missing_before_fill'] = pr['gender'].isna().astype(int)

    # Replace known placeholder age
    pr['age'] = pr['age'].replace(118, np.nan)
    pr['age_missing_before_impute'] = pr['age'].isna().astype(int)

    # Keep unknown gender as explicit segment
    pr['gender'] = pr['gender'].fillna('U').astype(str)

    # Marketing bands for segment-aware imputations
    pr['age_band_tmp'] = pd.cut(
        pr['age'],
        bins=[0, 25, 35, 45, 55, 65, np.inf],
        labels=['18_25', '26_35', '36_45', '46_55', '56_65', '66_plus']
    ).astype('object')
    pr['age_band_tmp'] = pr['age_band_tmp'].fillna('unknown')

    # Age imputation by gender median, then global median
    age_median_by_gender = pr.groupby('gender')['age'].median()
    pr['age'] = pr['age'].fillna(pr['gender'].map(age_median_by_gender))
    pr['age'] = pr['age'].fillna(pr['age'].median())

    # Income imputation by (gender, age band), then global median
    seg_median_income = pr.groupby(['gender', 'age_band_tmp'])['income'].transform('median')
    pr['income'] = pr['income'].fillna(seg_median_income)
    pr['income'] = pr['income'].fillna(pr['income'].median())

    # Final tidy
    pr['days_since_registration'] = pr['days_since_registration'].fillna(pr['days_since_registration'].median())
    pr['days_since_registration'] = pr['days_since_registration'].round().astype(int)
    pr['age'] = pr['age'].round().astype(int)

    pr = pr.drop(columns=['became_member_on'])

    return pr

profile_clean = clean_profile(profile_raw)
print('profile_clean:', profile_clean.shape)
profile_clean.head()

profile_clean: (17000, 10)


,gender,age,person,income,days_since_registration,age_was_118,income_missing_before_impute,gender_missing_before_fill,age_missing_before_impute,age_band_tmp
0,U,55,68be06ca386d4c31939f3a4f0e3dd783,64000.0,529,1,1,1,1,unknown
1,F,55,0610b486422d4921ae7d2bf64640c50b,112000.0,376,0,0,0,0,46_55
2,U,55,38fe809add3b4fcf9315a9694bb96ff5,64000.0,14,1,1,1,1,unknown
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,100000.0,443,0,0,0,0,66_plus
4,U,55,a03223e636434f42ac4c3df47e8bac43,64000.0,356,1,1,1,1,unknown


## Step 3) Clean `transcript` (Behavioral Event Table)

### Marketing rationale

`transcript` is behavior over time. For campaign analytics, sequence and timing matter more than simple counts.

Cleaning strategy:
- Extract normalized `offer_id`, `amount`, `reward` from nested `value` field.
- Keep event timeline fields for response logic.
- Build transaction aggregates (`transaction_count`, `total_spent`, `avg_transaction_value`) as customer value signals.
- Preserve one row per event for traceability, then derive person-offer journey in the next step.

In [27]:
def _extract_offer_id(value_obj):
    if isinstance(value_obj, dict):
        return value_obj.get('offer id') if value_obj.get('offer id') is not None else value_obj.get('offer_id')
    return np.nan

def _extract_amount(value_obj):
    if isinstance(value_obj, dict):
        return value_obj.get('amount', np.nan)
    return np.nan

def _extract_reward(value_obj):
    if isinstance(value_obj, dict):
        return value_obj.get('reward', np.nan)
    return np.nan

def clean_transcript(transcript_df: pd.DataFrame):
    t = transcript_df.copy()

    required_cols = ['person', 'event', 'time', 'value']
    missing = [c for c in required_cols if c not in t.columns]
    if missing:
        raise KeyError(f'transcript is missing required columns: {missing}')

    t['person'] = t['person'].astype(str)
    t['event'] = t['event'].astype(str).str.lower().str.strip()
    t['time'] = pd.to_numeric(t['time'], errors='coerce')

    t['offer_id'] = t['value'].apply(_extract_offer_id)
    t['amount'] = pd.to_numeric(t['value'].apply(_extract_amount), errors='coerce')
    t['reward_value'] = pd.to_numeric(t['value'].apply(_extract_reward), errors='coerce')

    # Aggregate transaction behavior as customer value features
    tx = t[t['event'] == 'transaction'].copy()
    transaction_features = (
        tx.groupby('person', as_index=False)
        .agg(
            transaction_count=('amount', 'count'),
            total_spent=('amount', 'sum'),
            avg_transaction_value=('amount', 'mean')
        )
    )

    # Fill absent transaction behavior with zeros later after merge
    return t, transaction_features

transcript_clean, transaction_features = clean_transcript(transcript_raw)
print('transcript_clean:', transcript_clean.shape)
print('transaction_features:', transaction_features.shape)
transcript_clean.head()

transcript_clean: (306534, 7)
transaction_features: (16578, 4)


,person,event,value,time,offer_id,amount,reward_value
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN


## Step 4) Build person-offer journey and target label

### Marketing rationale

A response label should reflect campaign exposure flow, not just presence of events.

Target logic used here (`offer_success`):
- Customer must have viewed the offer.
- Completion time must be after view time.
- Completion must occur within the offer duration window from receive time.

This creates a more realistic KPI for campaign effectiveness.

In [28]:
def build_offer_journey(transcript_df: pd.DataFrame, portfolio_df: pd.DataFrame) -> pd.DataFrame:
    offer_events = transcript_df[transcript_df['event'].isin(['offer received', 'offer viewed', 'offer completed'])].copy()
    offer_events = offer_events.dropna(subset=['offer_id'])
    offer_events['offer_id'] = offer_events['offer_id'].astype(str)

    # Earliest timestamp of each event by person-offer pair
    pivot = (
        offer_events.groupby(['person', 'offer_id', 'event'])['time']
        .min()
        .unstack('event')
        .reset_index()
    )

    for col in ['offer received', 'offer viewed', 'offer completed']:
        if col not in pivot.columns:
            pivot[col] = np.nan

    pivot = pivot.rename(columns={
        'offer received': 'time_received',
        'offer viewed': 'time_viewed',
        'offer completed': 'time_completed'
    })

    # Bring offer duration into journey table
    duration_table = portfolio_df[['offer_id', 'duration']].drop_duplicates()
    journey = pivot.merge(duration_table, on='offer_id', how='left')

    journey['was_viewed'] = journey['time_viewed'].notna().astype(int)
    journey['completed_after_view'] = (
        journey['time_completed'].notna() & journey['time_viewed'].notna() & (journey['time_completed'] >= journey['time_viewed'])
    )

    # Duration is in days while transcript time is in hours
    journey['within_offer_window'] = (
        journey['time_completed'].notna() & journey['time_received'].notna() & journey['duration'].notna() &
        ((journey['time_completed'] - journey['time_received']) <= (journey['duration'] * 24))
    )

    journey['offer_success'] = (
        (journey['was_viewed'] == 1) &
        (journey['completed_after_view']) &
        (journey['within_offer_window'])
    ).astype(int)

    # Track campaigns that were received but never viewed
    journey['received_not_viewed'] = (journey['time_received'].notna() & journey['time_viewed'].isna()).astype(int)

    return journey

journey_df = build_offer_journey(transcript_clean, portfolio_clean)
print('journey_df:', journey_df.shape)
print('offer_success rate (all received/viewed pairs):', round(journey_df['offer_success'].mean(), 4))
journey_df.head()

journey_df: (63288, 11)
offer_success rate (all received/viewed pairs): 0.3119


,person,offer_id,time_completed,time_received,time_viewed,duration,was_viewed,completed_after_view,within_offer_window,offer_success,received_not_viewed
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,NaN,7,0,False,True,0,1
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,NaN,336.0,372.0,4,1,False,False,0,0
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,NaN,168.0,192.0,3,1,False,False,0,0
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,5,1,False,True,0,0
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,10,1,False,True,0,0


## Step 5) Merge cleaned tables + apply marketing-aware missing strategy

### Why this is better than blind fillna

This step merges cleaned `journey`, `profile`, and `portfolio` tables and applies targeted preprocessing decisions:
- Keep unknown demographic groups explicit where behavior may differ.
- Use segment-informed imputations where numeric completeness is required.
- Preserve imputation flags for model explainability.
- Remove informational offers for conversion modeling consistency.

In [29]:
# Merge person-offer journey with customer and offer tables
df = journey_df.merge(profile_clean, on='person', how='left')
df = df.merge(portfolio_clean, on='offer_id', how='left')
df = df.merge(transaction_features, on='person', how='left')

# Normalize commonly used offer columns in case merge creates suffixes
for base_col in ['duration', 'difficulty', 'reward']:
    candidates = [c for c in [base_col, f'{base_col}_x', f'{base_col}_y'] if c in df.columns]
    if base_col not in df.columns and candidates:
        df[base_col] = df[candidates].bfill(axis=1).iloc[:, 0]
    extra_cols = [c for c in [f'{base_col}_x', f'{base_col}_y'] if c in df.columns and c != base_col]
    if extra_cols:
        df = df.drop(columns=extra_cols)

# Remove informational offers if one-hot encoded column exists
if 'offer_type_informational' in df.columns:
    df = df[df['offer_type_informational'] == 0].copy()

# Fill transaction behavior defaults: no transaction events -> zero behavior in observation window
for col in ['transaction_count', 'total_spent', 'avg_transaction_value']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Additional robust handling for any residual numeric gaps (rare, but safe for modeling)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if df[col].isna().any():
        df[f'{col}_was_imputed'] = df[col].isna().astype(int)
        df[col] = df[col].fillna(df[col].median())

# Create customer segment bins for marketing analysis
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 65, np.inf],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66+'],
    include_lowest=True
)
df['income_group'] = pd.cut(
    df['income'],
    bins=[0, 40000, 60000, 80000, 100000, np.inf],
    labels=['low', 'lower_middle', 'middle', 'upper_middle', 'high'],
    include_lowest=True
)

print('Merged modeling table shape:', df.shape)
print('Offer success rate:', round(df['offer_success'].mean(), 4))
print('Null-containing columns remaining:', int((df.isna().sum() > 0).sum()))
df.head()

Merged modeling table shape: (50637, 40)
Offer success rate: 0.3898
Null-containing columns remaining: 0


,person,offer_id,time_completed,time_received,time_viewed,was_viewed,completed_after_view,within_offer_window,offer_success,received_not_viewed,gender,age,income,days_since_registration,age_was_118,income_missing_before_impute,gender_missing_before_fill,age_missing_before_impute,age_band_tmp,reward,channels,difficulty,channel_email,channel_mobile,channel_social,channel_web,channel_count,reward_to_difficulty,reward_per_day,offer_type_bogo,offer_type_discount,offer_type_informational,transaction_count,total_spent,avg_transaction_value,duration,time_completed_was_imputed,time_viewed_was_imputed,age_group,income_group
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,360.0,0,False,True,0,1,M,33,72000.0,461,0,0,0,0,26_35,2,"[web, email, mobile]",10,1,1,0,1,3,0.20,0.285714,0,1,0,8.0,127.60,15.950000,7,0,1,26-35,middle
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,1,False,True,0,0,M,33,72000.0,461,0,0,0,0,26_35,5,"[web, email, mobile, social]",5,1,1,1,1,4,1.00,1.000000,1,0,0,8.0,127.60,15.950000,5,0,0,26-35,middle
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,1,False,True,0,0,M,33,72000.0,461,0,0,0,0,26_35,2,"[web, email, mobile, social]",10,1,1,1,1,4,0.20,0.200000,0,1,0,8.0,127.60,15.950000,10,0,0,26-35,middle
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,420.0,168.0,216.0,1,False,False,0,0,U,55,64000.0,92,1,1,1,1,unknown,5,"[web, email, mobile, social]",5,1,1,1,1,4,1.00,1.000000,1,0,0,3.0,4.09,1.363333,5,1,0,46-55,middle
6,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576.0,408.0,432.0,1,True,True,1,0,O,40,57000.0,198,0,0,0,0,36_45,5,"[web, email]",20,1,0,0,1,2,0.25,0.500000,0,1,0,5.0,79.46,15.892000,10,0,0,36-45,lower_middle


## Step 6) Add campaign and customer behavior features

These features are designed to improve both predictive performance and marketing interpretation:
- `membership_duration_months`: tenure scaled for segmentation.
- Channel preference flags from offer exposure.
- Spending intensity features (`spend_per_transaction`, `spend_per_membership_day`).
- Offer pressure feature (`difficulty_per_day`) and net incentive (`net_value_score`).

In [30]:
# Tenure in months
df['membership_duration_months'] = (df['days_since_registration'] / 30.44).round(2)

# Channel availability flags (aligns with 01_c style naming if channel columns exist)
for ch in ['web', 'email', 'mobile', 'social']:
    col = f'channel_{ch}'
    if col in df.columns:
        df[f'has_{ch}_channel'] = (df[col] > 0).astype(int)

# Spending behavior features
df['spend_per_transaction'] = np.where(
    df['transaction_count'] > 0,
    df['total_spent'] / df['transaction_count'],
    0.0
)
df['spend_per_membership_day'] = np.where(
    df['days_since_registration'] > 0,
    df['total_spent'] / df['days_since_registration'],
    0.0
)

# Offer pressure and value features
df['difficulty_per_day'] = np.where(df['duration'] > 0, df['difficulty'] / df['duration'], 0.0)
df['net_value_score'] = df['reward'] - (0.1 * df['difficulty'])

# Convert categorical bins to string for CSV compatibility
df['age_group'] = df['age_group'].astype(str)
df['income_group'] = df['income_group'].astype(str)

print('Feature engineering complete.')
print('Current shape:', df.shape)

Feature engineering complete.
Current shape: (50637, 49)


## Step 7) Final checks and export

Two outputs are saved:
- `preprocessed_data_rework_marketing.csv`: versioned output from this notebook.
- `preprocessed_data.csv`: default pipeline file for compatibility with downstream notebooks (including feature-adding notebooks).

In [31]:
# Keep a practical set of columns for modeling
drop_cols = ['channels', 'value', 'age_band_tmp']
for c in drop_cols:
    if c in df.columns:
        df = df.drop(columns=[c])

# Optional one-hot for remaining low-cardinality categoricals
categorical_to_encode = [c for c in ['gender'] if c in df.columns]
model_df = pd.get_dummies(df, columns=categorical_to_encode, drop_first=False)

# Final consistency checks
null_cols = model_df.columns[model_df.isna().any()].tolist()
print('Final model_df shape:', model_df.shape)
print('Target column exists:', 'offer_success' in model_df.columns)
print('Null columns:', null_cols)

output_dir = DATA_DIR / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

output_path_versioned = output_dir / 'preprocessed_data_rework_marketing.csv'
output_path_default = output_dir / 'preprocessed_data.csv'

model_df.to_csv(output_path_versioned, index=False)
model_df.to_csv(output_path_default, index=False)

print('Saved versioned file:', output_path_versioned)
print('Saved default file:', output_path_default)
model_df.head()

Final model_df shape: (50637, 50)
Target column exists: True
Null columns: []
Saved versioned file: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\processed\preprocessed_data_rework_marketing.csv
Saved default file: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\processed\preprocessed_data.csv


,person,offer_id,time_completed,time_received,time_viewed,was_viewed,completed_after_view,within_offer_window,offer_success,received_not_viewed,age,income,days_since_registration,age_was_118,income_missing_before_impute,gender_missing_before_fill,age_missing_before_impute,reward,difficulty,channel_email,channel_mobile,channel_social,channel_web,channel_count,reward_to_difficulty,reward_per_day,offer_type_bogo,offer_type_discount,offer_type_informational,transaction_count,total_spent,avg_transaction_value,duration,time_completed_was_imputed,time_viewed_was_imputed,age_group,income_group,membership_duration_months,has_web_channel,has_email_channel,has_mobile_channel,has_social_channel,spend_per_transaction,spend_per_membership_day,difficulty_per_day,net_value_score,gender_F,gender_M,gender_O,gender_U
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,360.0,0,False,True,0,1,33,72000.0,461,0,0,0,0,2,10,1,1,0,1,3,0.20,0.285714,0,1,0,8.0,127.60,15.950000,7,0,1,26-35,middle,15.14,1,1,1,0,15.950000,0.276790,1.428571,1.0,False,True,False,False
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,8.0,127.60,15.950000,5,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,4.5,False,True,False,False
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,2,10,1,1,1,1,4,0.20,0.200000,0,1,0,8.0,127.60,15.950000,10,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,1.0,False,True,False,False
5,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,420.0,168.0,216.0,1,False,False,0,0,55,64000.0,92,1,1,1,1,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,3.0,4.09,1.363333,5,1,0,46-55,middle,3.02,1,1,1,1,1.363333,0.044457,1.000000,4.5,False,False,False,True
6,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576.0,408.0,432.0,1,True,True,1,0,40,57000.0,198,0,0,0,0,5,20,1,0,0,1,2,0.25,0.500000,0,1,0,5.0,79.46,15.892000,10,0,0,36-45,lower_middle,6.50,1,1,0,0,15.892000,0.401313,2.000000,3.0,False,False,True,False
